In [2]:
import pandas as pd
df = pd.read_csv('/content/train[2].csv')

In [3]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = df.drop(columns=['Cabin'])

In [4]:
df = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]

In [5]:
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

In [6]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [13]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8100558659217877


In [15]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[90 15]
 [19 55]]


In [17]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



## Why Accuracy Alone Can Be Misleading

Accuracy does not always show the true performance of a model, especially when the dataset is imbalanced. A model may achieve high accuracy by predicting only the majority class while failing to identify the minority class correctly. Precision, Recall, and F1-score provide a better understanding of the model's performance by evaluating different aspects of classification.

In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Cross Validation Score:", grid.best_score_)

Best Parameters: {'C': 1, 'solver': 'liblinear'}
Best Cross Validation Score: 0.7962966610853935


In [20]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Tuned Accuracy:", accuracy_score(y_test, y_pred_best))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best))

Tuned Accuracy: 0.7821229050279329

Classification Report:

              precision    recall  f1-score   support

           0       0.79      0.85      0.82       105
           1       0.76      0.69      0.72        74

    accuracy                           0.78       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179



In [21]:
original_accuracy = accuracy_score(y_test, y_pred)
tuned_accuracy = accuracy_score(y_test, y_pred_best)

comparison = pd.DataFrame({
    "Model": ["Original Logistic Regression", "Tuned Logistic Regression"],
    "Accuracy": [original_accuracy, tuned_accuracy]
})

comparison

,Model,Accuracy
0,Original Logistic Regression,0.810056
1,Tuned Logistic Regression,0.782123


## Hyperparameter Tuning Result

The Logistic Regression model was tuned using GridSearchCV with different values of C and solver.

The tuned model achieved an accuracy of 78.21%, while the original model achieved 81.01%.

This shows that hyperparameter tuning does not always improve performance. In this case, the original model performed better on the test dataset.